# Length of Hospitalisation Sensitivity Analysis

This notebook is a sensitivity analysis only. LENGTH OF HOSPITALISATION is intentionally included to explore its SHAP behaviour and possible illness-course signal. It should not be interpreted as part of the main deployment-valid prediction model because it may represent post-admission/post-outcome leakage.

**Sensitivity model, not main model.** The main EEG abnormality prediction model should still exclude `LENGTH OF HOSPITALISATION`.


# 1. Data Loading

This section defines the EEG project context, imports libraries, loads the HCTM dataset, and constructs the binary EEG abnormality target.

## 1.1 Project Setup

In [ ]:
# Core libraries
import os
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
MAIN_OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR = MAIN_OUTPUT_DIR / "length_of_hospitalisation_sensitivity"
SHAP_OUTPUT_DIR = OUTPUT_DIR / "shap"
MPLCONFIG_DIR = OUTPUT_DIR / "matplotlib"
for directory in [OUTPUT_DIR, SHAP_OUTPUT_DIR, MPLCONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(MPLCONFIG_DIR)
os.environ["PYTHONHASHSEED"] = "42"
warnings.filterwarnings("ignore")

import numpy as np
if not hasattr(np, "NaN"):
    np.NaN = np.nan
import pandas as pd
import scipy.stats as scipy_stats

if not hasattr(scipy_stats, "binom_test"):
    scipy_stats.binom_test = lambda x, n=None, p=0.5, alternative="two-sided": scipy_stats.binomtest(
        int(round(x)), n=int(n), p=p, alternative=alternative
    ).pvalue

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

from boruta import BorutaPy
from BorutaShap import BorutaShap
from catboost import CatBoostClassifier, Pool
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print(f"Project root: {PROJECT_ROOT}")
print(f"Main-model output directory for comparison: {MAIN_OUTPUT_DIR}")
print(f"Sensitivity output directory: {OUTPUT_DIR}")
print(f"Random seed: {RANDOM_STATE}")

## 1.2 Project Configuration

Project-specific constants are defined here. The remaining notebook follows the healthcare ML template workflow.

In [ ]:
PROJECT_NAME = "EEG Abnormality Prediction - Length of Hospitalisation Sensitivity"
SHORT_NAME = "EEG Abnormality Length of Hospitalisation Sensitivity"
DATA_PATH = Path("Dataset/EEG ABNORMALITY DATASET.xlsx")
SHEET_NAME = "Raw Data Set"
TARGET_COLUMN = "EEG FINDINGS"
TARGET_BINARY = "EEG_ABNORMAL"
POSITIVE_CLASS_LABEL = "Abnormal EEG"
SENSITIVITY_FEATURE = "LENGTH OF HOSPITALISATION"
SENSITIVITY_LABEL = "Sensitivity model, not main model"
SENSITIVITY_WARNING = (
    "This notebook is a sensitivity analysis only. LENGTH OF HOSPITALISATION is intentionally included "
    "to explore its SHAP behaviour and possible illness-course signal. It should not be interpreted as "
    "part of the main deployment-valid prediction model because it may represent post-admission/post-outcome leakage."
)
TEST_SIZE = 0.20
CV_FOLDS = 5
BORUTASHAP_N_TRIALS = 100
BORUTASHAP_PERCENTILE = 100
BORUTASHAP_PVALUE = 0.05
BORUTASHAP_CATBOOST_ITERATIONS = 80

DATASET_SOURCE = "Hospital Canselor Tuanku Muhriz (HCTM)"
RESEARCH_ONLY_NOTE = (
    "Research-only retrospective single-centre analysis. "
    "No clinical deployment, diagnosis, or treatment decision claims are made."
)

config_summary = pd.DataFrame(
    {
        "item": [
            "project",
            "dataset_source",
            "dataset_path",
            "sheet_name",
            "target_column",
            "binary_target",
            "positive_class",
            "analysis_label",
            "sensitivity_feature",
            "test_size",
            "cv_folds",
        ],
        "value": [
            PROJECT_NAME,
            DATASET_SOURCE,
            str(DATA_PATH),
            SHEET_NAME,
            TARGET_COLUMN,
            TARGET_BINARY,
            POSITIVE_CLASS_LABEL,
            SENSITIVITY_LABEL,
            SENSITIVITY_FEATURE,
            TEST_SIZE,
            CV_FOLDS,
        ],
    }
)
display(config_summary)

## 1.3 Load Dataset

In [ ]:
def load_dataset(path, sheet_name=None):
    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported dataset suffix: {suffix}")


if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

raw_data = load_dataset(DATA_PATH, SHEET_NAME)
raw_shape = raw_data.shape
raw_columns_original = list(raw_data.columns)
raw_data = raw_data.copy()
raw_data.columns = raw_data.columns.astype(str).str.strip()
raw_columns_cleaned = list(raw_data.columns)

column_check = pd.DataFrame(
    {
        "column": raw_columns_cleaned,
        "dtype": [str(raw_data[col].dtype) for col in raw_columns_cleaned],
        "missing_n": [int(raw_data[col].isna().sum()) for col in raw_columns_cleaned],
        "missing_pct": [float(raw_data[col].isna().mean() * 100) for col in raw_columns_cleaned],
    }
)

changed_columns = [before for before, after in zip(raw_columns_original, raw_columns_cleaned) if before != after]
print(f"Raw dataset shape: {raw_shape[0]} rows x {raw_shape[1]} columns")
print("Column-name whitespace was stripped in memory only; the raw workbook was not edited.")
print(f"Columns changed after stripping whitespace: {changed_columns}")
display(column_check)

## 1.4 Target Definition

The target maps `Normal` EEG findings to 0 and all clinically non-normal EEG findings to 1. Rows with missing or invalid target values are removed before mapping.

In [ ]:
data = raw_data.copy()

if TARGET_COLUMN not in data.columns:
    raise KeyError(f"Required target column not found: {TARGET_COLUMN}")


def clean_text_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in {"", "-", "NA", "N/A", "None", "NULL", "nan"}:
        return np.nan
    return text


target_clean = data[TARGET_COLUMN].map(clean_text_value)
valid_target_mask = target_clean.notna()
invalid_target_rows = int((~valid_target_mask).sum())
if invalid_target_rows:
    print(f"Dropping {invalid_target_rows} rows with missing/invalid target values.")

data = data.loc[valid_target_mask].copy()
target_clean = target_clean.loc[valid_target_mask]
data[TARGET_BINARY] = np.where(target_clean.str.lower().eq("normal"), 0, 1).astype(int)

raw_target_distribution = target_clean.value_counts(dropna=False).rename_axis("raw_eeg_finding").reset_index(name="n")
binary_target_distribution = (
    data[TARGET_BINARY]
    .map({0: "Normal EEG (0)", 1: "Abnormal EEG (1)"})
    .value_counts()
    .rename_axis("target_class")
    .reset_index(name="n")
)
binary_target_distribution["pct"] = binary_target_distribution["n"] / binary_target_distribution["n"].sum() * 100

print(f"Analysis dataset shape after target cleaning: {data.shape[0]} rows x {data.shape[1]} columns")
display(raw_target_distribution)
display(binary_target_distribution)

# 2. Preprocessing

This section implements leakage-aware feature exclusion, EDA-driven cleaning, stratified splitting, train-fitted preprocessing, and model-specific feature selection before benchmarking.

## 2.1 Feature Exclusion

The following exclusions remove leakage, downstream outcomes, post-hoc interpretations, unavailable fields, high-missing free text, and redundant raw fields. This is feature exclusion, not feature selection.

In [ ]:
LEAKAGE_COLUMNS = [
    "CAUSES",
    "OUTCOME OF HOSPITALISATION",
    "INDICATION FOR EEG",
    "DIAGNOSIS",
    "DXT/FBS/RBS/HbA1c",
    "OTHERS",
    "TOTAL CHOLESTEROL",
    "BLOOD PRESSURE ON PRESENTATION",
]

exclusion_reasons = {
    "CAUSES": "Leakage / diagnostic pathway / possible post-hoc clinical interpretation",
    "OUTCOME OF HOSPITALISATION": "Post-outcome",
    "INDICATION FOR EEG": "EEG-ordering pathway / potential target proxy",
    "DIAGNOSIS": "Target proxy / post-clinical interpretation",
    "DXT/FBS/RBS/HbA1c": "Excluded raw/related glucose measurement field",
    "OTHERS": "99 percent missing / free-text / low utility",
    "TOTAL CHOLESTEROL": "Excluded based on prior redundancy/project decision",
    "BLOOD PRESSURE ON PRESENTATION": "Raw combined BP field; separated BP fields are used if retained",
}

present_leakage_columns = [col for col in LEAKAGE_COLUMNS if col in data.columns]
missing_leakage_columns = [col for col in LEAKAGE_COLUMNS if col not in data.columns]

sensitivity_included_columns = [SENSITIVITY_FEATURE] if SENSITIVITY_FEATURE in data.columns else []
sensitivity_inclusion_summary = pd.DataFrame(
    [
        {
            "column": SENSITIVITY_FEATURE,
            "present_in_dataset": SENSITIVITY_FEATURE in data.columns,
            "sensitivity_decision": "Intentionally retained as candidate predictor for sensitivity analysis only",
            "main_model_decision": "Excluded from main model as post-admission/post-outcome leakage risk",
        }
    ]
)

exclusion_summary = pd.DataFrame(
    {
        "column": LEAKAGE_COLUMNS,
        "present_in_dataset": [col in data.columns for col in LEAKAGE_COLUMNS],
        "reason": [exclusion_reasons[col] for col in LEAKAGE_COLUMNS],
    }
)

display(exclusion_summary)
display(sensitivity_inclusion_summary)
print(SENSITIVITY_WARNING)
print(f"Excluded leakage/downstream/unavailable columns present: {len(present_leakage_columns)}")
print(f"Specified exclusion columns not present: {missing_leakage_columns}")


## 2.2 Feature Type Split/Definition

Feature groups are defined explicitly so demographic categories, binary clinical status variables, laboratory values, and CatBoost categorical columns are handled according to the project context.

In [ ]:
numeric_columns_spec = [
    "AGE",
    "Systolic",
    "Diastolic",
    "LENGTH OF HOSPITALISATION",
    "HAEMOGLOBIN",
    "PLATELET COUNT",
    "WHITE CELL COUNT",
    "BLOOD UREA",
    "POTASSIUM",
    "CREATININE",
    "SODIUM",
    "TOTAL PROTEIN",
    "ALBUMIN",
    "ALANINE TRANSAMINASE",
    "CALCIUM",
    "TRIGLYCERIDE(TG)",
    "LDL",
    "HDL",
]

binary_clinical_categorical_spec = [
    "SMOKING",
    "ALCOHOL",
    "HYPERTENSION",
    "DIABETES MELLITUS",
    "HYPERLIPIDEMIA",
    "STROKE",
    "TB",
    "AF",
    "IHD",
    "CONGESTIVE CARDIAC FAILURE",
    "COPD",
    "BRONCHIAL ASTHMA",
    "CKD",
    "PSYCHIATRIC DISORDER",
    "BRAIN TUMOUR",
    "HIV",
    "SEIZURES",
    "FOCAL SEIZURE",
    "GENERALISED SEIZURE",
]

demographic_categorical_spec = ["GENDER", "RACE"]
nonbinary_clinical_categorical_spec = ["Blood Sugar Control", "RADIOLOGICAL IMAGING"]

numeric_columns = [col for col in numeric_columns_spec if col in data.columns]
binary_clinical_categorical_columns = [col for col in binary_clinical_categorical_spec if col in data.columns]
demographic_categorical_columns = [col for col in demographic_categorical_spec if col in data.columns]
nonbinary_clinical_categorical_columns = [col for col in nonbinary_clinical_categorical_spec if col in data.columns]
categorical_columns = (
    binary_clinical_categorical_columns
    + demographic_categorical_columns
    + nonbinary_clinical_categorical_columns
)

model_data = data.drop(columns=present_leakage_columns, errors="ignore").copy()
feature_columns = [
    col
    for col in numeric_columns + categorical_columns
    if col in model_data.columns and col not in {TARGET_COLUMN, TARGET_BINARY}
]

missing_expected = sorted(
    set(numeric_columns_spec + binary_clinical_categorical_spec + demographic_categorical_spec + nonbinary_clinical_categorical_spec)
    - set(feature_columns)
    - set(LEAKAGE_COLUMNS)
)

feature_type_summary = pd.DataFrame(
    [
        {"feature_group": "numerical", "n": len(numeric_columns), "features": numeric_columns},
        {
            "feature_group": "binary_clinical_categorical",
            "n": len(binary_clinical_categorical_columns),
            "features": binary_clinical_categorical_columns,
        },
        {"feature_group": "demographic_categorical", "n": len(demographic_categorical_columns), "features": demographic_categorical_columns},
        {
            "feature_group": "nonbinary_clinical_categorical",
            "n": len(nonbinary_clinical_categorical_columns),
            "features": nonbinary_clinical_categorical_columns,
        },
    ]
)

print(f"Retained modeling features before feature selection: {len(feature_columns)}")
print(f"{SENSITIVITY_LABEL}: {SENSITIVITY_FEATURE} retained as candidate predictor = {SENSITIVITY_FEATURE in feature_columns}")
print(f"Missing expected retained features after in-memory column cleanup: {missing_expected}")
display(feature_type_summary)


## 2.3 EDA-Driven Data Cleaning

The missingness and outlier policies below are implemented using training-set fitted values only. Medical outliers are not deleted blindly.

In [ ]:
numerical_missingness_report = pd.DataFrame(
    [
        ("HAEMOGLOBIN", 3, 1.041667),
        ("PLATELET COUNT", 3, 1.041667),
        ("WHITE CELL COUNT", 3, 1.041667),
        ("BLOOD UREA", 4, 1.388889),
        ("POTASSIUM", 2, 0.694444),
        ("CREATININE", 2, 0.694444),
        ("SODIUM", 2, 0.694444),
        ("TOTAL PROTEIN", 5, 1.736111),
        ("ALBUMIN", 3, 1.041667),
        ("ALANINE TRANSAMINASE", 4, 1.388889),
        ("CALCIUM", 4, 1.388889),
        ("TRIGLYCERIDE(TG)", 52, 18.055556),
        ("LDL", 51, 17.708333),
        ("HDL", 52, 18.055556),
    ],
    columns=["feature", "missing_values", "missing_pct"],
)

categorical_missingness_report = pd.DataFrame(
    [
        ("RADIOLOGICAL IMAGING", 13, 4.513889, "Candidate predictor; preserve imaging findings / Not performed category"),
        ("Blood Sugar Control", 42, 14.583333, "Missing/not tested/not documented as 2"),
        ("DXT/FBS/RBS/HbA1c", 40, 13.888889, "Excluded"),
        ("OTHERS", 286, 99.305556, "Excluded"),
    ],
    columns=["feature", "missing_values", "missing_pct", "handling"],
)

outlier_policy = pd.DataFrame(
    [
        ("AGE", "Retain clinically plausible values"),
        ("Systolic", "Retain clinically plausible values"),
        ("Diastolic", "Retain clinically plausible values"),
        ("PLATELET COUNT", "Train-derived tail capping and log transform if skewed"),
        ("WHITE CELL COUNT", "Train-derived tail capping and log transform if skewed"),
        ("BLOOD UREA", "Train-derived tail capping and log transform if skewed"),
        ("CREATININE", "Train-derived tail capping and log transform if skewed"),
        ("ALANINE TRANSAMINASE", "Train-derived tail capping and log transform if skewed"),
        ("TRIGLYCERIDE(TG)", "Missingness flag, median imputation, log transform if skewed"),
        ("LDL", "Missingness flag and retain/cap only if invalid"),
        ("HDL", "Missingness flag and retain/cap only if invalid"),
    ],
    columns=["feature", "handling"],
)

missingness_policy = pd.DataFrame(
    [
        ("Low-missing numerical features", "Training-set median"),
        ("TRIGLYCERIDE(TG), LDL, HDL", "Training-set median plus missingness flag"),
        ("Binary clinical categorical features", "Missing/unmapped as 2 where clinically appropriate"),
        ("Demographic categorical features", "Training-set mode imputation if missing"),
        ("Blood Sugar Control", "Missing/unknown/not tested as 2"),
        ("RADIOLOGICAL IMAGING", "Candidate predictor; preserve imaging findings and missing/not performed category"),
    ],
    columns=["feature_or_group", "handling_rule"],
)

display(numerical_missingness_report)
display(categorical_missingness_report)
display(outlier_policy)
display(missingness_policy)


In [ ]:
MISSING_TOKENS = {"", " ", "-", "NA", "N/A", "na", "n/a", "None", "none", "NULL", "null", "nan", "NaN"}
YES_TOKENS = {"yes", "y", "true", "1", "present", "positive"}
NO_TOKENS = {"no", "n", "false", "0", "absent", "negative"}
UNKNOWN_TOKENS = {"unknown", "not known", "not documented", "not done", "not performed", "not tested", "not applicable", "missing"}

LIPID_MISSING_FLAG_FEATURES = ["TRIGLYCERIDE(TG)", "LDL", "HDL"]
CAPPING_CANDIDATES = ["PLATELET COUNT", "WHITE CELL COUNT", "BLOOD UREA", "CREATININE", "ALANINE TRANSAMINASE"]
LOG_TRANSFORM_CANDIDATES = CAPPING_CANDIDATES + ["TRIGLYCERIDE(TG)"]


def normalize_missing_tokens(df):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].map(lambda value: np.nan if pd.isna(value) or str(value).strip() in MISSING_TOKENS else str(value).strip())
    return df


def coerce_numeric(series):
    cleaned = series.map(lambda value: np.nan if pd.isna(value) or str(value).strip() in MISSING_TOKENS else value)
    cleaned = cleaned.astype("string").str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce").astype(float)


def map_binary_clinical_value(value):
    if pd.isna(value):
        return "2"
    text = str(value).strip().lower()
    if text in MISSING_TOKENS or text in UNKNOWN_TOKENS:
        return "2"
    if text in YES_TOKENS:
        return "1"
    if text in NO_TOKENS:
        return "0"
    return "2"


def clean_category_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in MISSING_TOKENS or text.lower() in UNKNOWN_TOKENS:
        return np.nan
    return text


class ClinicalPreprocessor:
    def __init__(
        self,
        numeric_columns,
        binary_categorical_columns,
        demographic_categorical_columns,
        nonbinary_categorical_columns,
        missing_flag_features,
        capping_candidates,
        log_transform_candidates,
    ):
        self.numeric_columns = list(numeric_columns)
        self.binary_categorical_columns = list(binary_categorical_columns)
        self.demographic_categorical_columns = list(demographic_categorical_columns)
        self.nonbinary_categorical_columns = list(nonbinary_categorical_columns)
        self.missing_flag_features = [col for col in missing_flag_features if col in self.numeric_columns]
        self.missing_flag_columns = [f"{col}__missing_flag" for col in self.missing_flag_features]
        self.capping_candidates = [col for col in capping_candidates if col in self.numeric_columns]
        self.log_transform_candidates = [col for col in log_transform_candidates if col in self.numeric_columns]
        self.numeric_medians_ = {}
        self.categorical_modes_ = {}
        self.cap_bounds_ = {}
        self.log_transform_columns_ = []

    def fit(self, X, y=None):
        X_norm = normalize_missing_tokens(X)
        numeric_train = pd.DataFrame(index=X_norm.index)
        for col in self.numeric_columns:
            numeric_train[col] = coerce_numeric(X_norm[col])
            median = numeric_train[col].median()
            self.numeric_medians_[col] = 0.0 if pd.isna(median) else float(median)
            if col in self.capping_candidates:
                observed = numeric_train[col].dropna()
                lower = self.numeric_medians_[col] if observed.empty else float(observed.quantile(0.01))
                upper = self.numeric_medians_[col] if observed.empty else float(observed.quantile(0.99))
                if lower > upper:
                    lower, upper = upper, lower
                self.cap_bounds_[col] = (lower, upper)

        for col in self.demographic_categorical_columns:
            cleaned = X_norm[col].map(clean_category_value)
            mode_values = cleaned.dropna().mode()
            self.categorical_modes_[col] = "Unknown" if mode_values.empty else str(mode_values.iloc[0])

        numeric_for_skew = numeric_train.copy()
        for col in self.numeric_columns:
            numeric_for_skew[col] = numeric_for_skew[col].fillna(self.numeric_medians_[col])
            if col in self.cap_bounds_:
                lower, upper = self.cap_bounds_[col]
                numeric_for_skew[col] = numeric_for_skew[col].clip(lower=lower, upper=upper)

        self.log_transform_columns_ = [
            col
            for col in self.log_transform_candidates
            if numeric_for_skew[col].min() >= 0 and abs(numeric_for_skew[col].skew()) >= 1.0
        ]
        self.numeric_model_columns_ = self.numeric_columns + self.missing_flag_columns
        self.categorical_model_columns_ = (
            self.binary_categorical_columns
            + self.demographic_categorical_columns
            + self.nonbinary_categorical_columns
        )
        self.feature_order_ = self.numeric_model_columns_ + self.categorical_model_columns_
        return self

    def transform(self, X):
        X_norm = normalize_missing_tokens(X)
        transformed = pd.DataFrame(index=X_norm.index)
        for col in self.numeric_columns:
            numeric = coerce_numeric(X_norm[col])
            if col in self.missing_flag_features:
                transformed[f"{col}__missing_flag"] = numeric.isna().astype(int)
            numeric = numeric.fillna(self.numeric_medians_[col])
            if col in self.cap_bounds_:
                lower, upper = self.cap_bounds_[col]
                numeric = numeric.clip(lower=lower, upper=upper)
            if col in self.log_transform_columns_:
                numeric = np.log1p(numeric.clip(lower=0))
            transformed[col] = numeric.astype(float)
        for col in self.binary_categorical_columns:
            transformed[col] = X_norm[col].map(map_binary_clinical_value).astype(str)
        for col in self.demographic_categorical_columns:
            transformed[col] = X_norm[col].map(clean_category_value).fillna(self.categorical_modes_[col]).astype(str)
        for col in self.nonbinary_categorical_columns:
            transformed[col] = X_norm[col].map(clean_category_value).fillna("2").astype(str)
        return transformed[self.feature_order_].copy()

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


print("Cleaning helpers prepared.")

## 2.4 Stratified Train-Test Split

The split is performed before fitting imputation, capping, transformations, encoders, scalers, feature selectors, tuning, or threshold selection.

In [ ]:
X = model_data[feature_columns].copy()
y = model_data[TARGET_BINARY].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    [
        {"split": "train", "n": len(y_train), "abnormal_n": int(y_train.sum()), "abnormal_pct": float(y_train.mean() * 100)},
        {"split": "held_out_test", "n": len(y_test), "abnormal_n": int(y_test.sum()), "abnormal_pct": float(y_test.mean() * 100)},
    ]
)
print("Stratified split complete. Held-out test data is reserved for final evaluation only.")
display(split_summary)

## 2.5 Preprocessing Pipeline Execution

A shared clinical cleaner is fit on the training set, then model-specific matrices are created for linear, tree, and CatBoost models.

In [ ]:
def make_ohe_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


class ModelMatrixBuilder:
    def __init__(self, numeric_columns, categorical_columns):
        self.numeric_columns = list(numeric_columns)
        self.categorical_columns = list(categorical_columns)
        self.scaler_ = StandardScaler()
        self.ohe_ = make_ohe_encoder()

    def fit(self, X_clean):
        self.scaler_.fit(X_clean[self.numeric_columns])
        self.ohe_.fit(X_clean[self.categorical_columns])
        self.encoded_categorical_columns_ = list(self.ohe_.get_feature_names_out(self.categorical_columns))
        self.linear_feature_names_ = self.numeric_columns + self.encoded_categorical_columns_
        self.tree_feature_names_ = self.numeric_columns + self.encoded_categorical_columns_
        return self

    def transform_linear(self, X_clean):
        numeric_scaled = pd.DataFrame(
            self.scaler_.transform(X_clean[self.numeric_columns]),
            columns=self.numeric_columns,
            index=X_clean.index,
        )
        categorical_encoded = pd.DataFrame(
            self.ohe_.transform(X_clean[self.categorical_columns]),
            columns=self.encoded_categorical_columns_,
            index=X_clean.index,
        )
        return pd.concat([numeric_scaled, categorical_encoded], axis=1)[self.linear_feature_names_]

    def transform_tree(self, X_clean):
        numeric_unscaled = X_clean[self.numeric_columns].astype(float).copy()
        categorical_encoded = pd.DataFrame(
            self.ohe_.transform(X_clean[self.categorical_columns]),
            columns=self.encoded_categorical_columns_,
            index=X_clean.index,
        )
        return pd.concat([numeric_unscaled, categorical_encoded], axis=1)[self.tree_feature_names_]


def prepare_catboost_frame(X_clean, categorical_columns):
    X_cb = X_clean.copy()
    for col in categorical_columns:
        X_cb[col] = X_cb[col].astype(str)
    return X_cb


print("Model matrix builder prepared.")

In [ ]:
clinical_preprocessor = ClinicalPreprocessor(
    numeric_columns=numeric_columns,
    binary_categorical_columns=binary_clinical_categorical_columns,
    demographic_categorical_columns=demographic_categorical_columns,
    nonbinary_categorical_columns=nonbinary_clinical_categorical_columns,
    missing_flag_features=LIPID_MISSING_FLAG_FEATURES,
    capping_candidates=CAPPING_CANDIDATES,
    log_transform_candidates=LOG_TRANSFORM_CANDIDATES,
)

X_train_clean = clinical_preprocessor.fit_transform(X_train_raw, y_train)
X_test_clean = clinical_preprocessor.transform(X_test_raw)

numeric_model_columns = clinical_preprocessor.numeric_model_columns_
categorical_model_columns = clinical_preprocessor.categorical_model_columns_
catboost_categorical_features = list(categorical_model_columns)

matrix_builder = ModelMatrixBuilder(numeric_model_columns, categorical_model_columns).fit(X_train_clean)
X_train_linear = matrix_builder.transform_linear(X_train_clean)
X_test_linear = matrix_builder.transform_linear(X_test_clean)
X_train_tree = matrix_builder.transform_tree(X_train_clean)
X_test_tree = matrix_builder.transform_tree(X_test_clean)
X_train_catboost = prepare_catboost_frame(X_train_clean, catboost_categorical_features)
X_test_catboost = prepare_catboost_frame(X_test_clean, catboost_categorical_features)

preprocessing_summary = pd.DataFrame(
    [
        {"artifact": "clean_train_shape", "value": X_train_clean.shape},
        {"artifact": "linear_train_shape", "value": X_train_linear.shape},
        {"artifact": "tree_train_shape", "value": X_train_tree.shape},
        {"artifact": "catboost_train_shape", "value": X_train_catboost.shape},
        {"artifact": "numeric_model_columns", "value": len(numeric_model_columns)},
        {"artifact": "categorical_model_columns", "value": len(categorical_model_columns)},
        {"artifact": "log_transformed_columns", "value": clinical_preprocessor.log_transform_columns_},
        {"artifact": "train_derived_cap_bounds", "value": clinical_preprocessor.cap_bounds_},
    ]
)
display(preprocessing_summary)

## 2.6 Feature Selection

Feature selection is performed before benchmarking and uses training data only. Feature exclusion above is not counted as feature selection.

CatBoost uses BorutaSHAP for all-relevant ML risk-factor discovery. Tentative BorutaSHAP features are reported separately and are not included in the main CatBoost model unless explicitly promoted by a predefined rule.

In [ ]:
FEATURE_SELECTION_METHODS = {
    "logistic_regression": "L1/Lasso LogisticRegressionCV on scaled one-hot training matrix",
    "linear_svm": "RFECV with LinearSVC on scaled one-hot training matrix",
    "random_forest": "Boruta with RandomForestClassifier on unscaled one-hot training matrix",
    "catboost": "BorutaSHAP all-relevant feature selection with CatBoostClassifier on train-only CatBoost matrix",
}

selected_features = {}
selected_feature_counts = {}
feature_selection_details = []
selected_catboost_categorical_features = []
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

display(pd.DataFrame([{"model": key, "method": value} for key, value in FEATURE_SELECTION_METHODS.items()]))

In [ ]:
l1_selector = LogisticRegressionCV(
    penalty="l1",
    solver="liblinear",
    Cs=np.logspace(-3, 1, 10),
    cv=cv,
    scoring="f1",
    class_weight="balanced",
    max_iter=5000,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
l1_selector.fit(X_train_linear, y_train)
lr_selected = list(X_train_linear.columns[np.abs(l1_selector.coef_).ravel() > 1e-8])
if not lr_selected:
    raise RuntimeError("L1/Lasso Logistic Regression selected zero features; stopping rather than benchmarking full features.")
selected_features["logistic_regression"] = lr_selected
feature_selection_details.append(
    {
        "model": "Logistic Regression",
        "method": FEATURE_SELECTION_METHODS["logistic_regression"],
        "selected_count": len(lr_selected),
        "details": f"Best C={float(l1_selector.C_[0]):.5g}",
    }
)

linear_svm_rfecv = RFECV(
    estimator=LinearSVC(penalty="l2", class_weight="balanced", max_iter=30000, dual="auto", random_state=RANDOM_STATE),
    step=0.1,
    min_features_to_select=min(5, X_train_linear.shape[1]),
    cv=cv,
    scoring="f1",
    n_jobs=1,
)
linear_svm_rfecv.fit(X_train_linear, y_train)
svm_selected = list(X_train_linear.columns[linear_svm_rfecv.support_])
if not svm_selected:
    raise RuntimeError("RFECV Linear SVM selected zero features; stopping rather than benchmarking full features.")
selected_features["linear_svm"] = svm_selected
feature_selection_details.append(
    {
        "model": "Linear SVM",
        "method": FEATURE_SELECTION_METHODS["linear_svm"],
        "selected_count": len(svm_selected),
        "details": f"RFECV selected {linear_svm_rfecv.n_features_} features",
    }
)

rf_for_boruta = RandomForestClassifier(n_estimators=1000, max_depth=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)
boruta_selector = BorutaPy(
    estimator=rf_for_boruta,
    n_estimators="auto",
    perc=100,
    alpha=0.05,
    two_step=True,
    max_iter=100,
    random_state=RANDOM_STATE,
    verbose=0,
)
boruta_selector.fit(X_train_tree.values, y_train.values)
rf_selected_confirmed = list(X_train_tree.columns[boruta_selector.support_])
rf_selected_tentative = list(X_train_tree.columns[boruta_selector.support_weak_])
if not rf_selected_confirmed:
    raise RuntimeError(
        "Boruta selected zero confirmed Random Forest features; stopping rather than benchmarking Random Forest with full features. "
        f"Tentative features: {rf_selected_tentative}"
    )
selected_features["random_forest"] = rf_selected_confirmed
feature_selection_details.append(
    {
        "model": "Random Forest",
        "method": FEATURE_SELECTION_METHODS["random_forest"],
        "selected_count": len(rf_selected_confirmed),
        "details": f"Tentative features: {len(rf_selected_tentative)}",
    }
)

borutashap_catboost_model = CatBoostClassifier(
    iterations=BORUTASHAP_CATBOOST_ITERATIONS,
    depth=4,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)

X_train_borutashap = X_train_catboost.reset_index(drop=True).copy()
for col in catboost_categorical_features:
    if col in X_train_borutashap.columns:
        X_train_borutashap[col] = X_train_borutashap[col].astype("category")
y_train_borutashap = y_train.reset_index(drop=True).astype(int)

borutashap_selector = BorutaShap(
    model=borutashap_catboost_model,
    importance_measure="shap",
    classification=True,
    percentile=BORUTASHAP_PERCENTILE,
    pvalue=BORUTASHAP_PVALUE,
)
borutashap_selector.fit(
    X=X_train_borutashap,
    y=y_train_borutashap,
    n_trials=BORUTASHAP_N_TRIALS,
    random_state=RANDOM_STATE,
    sample=False,
    train_or_test="train",
    normalize=True,
    verbose=False,
)

borutashap_accepted_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.accepted)]
borutashap_tentative_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.tentative)]
borutashap_rejected_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.rejected)]
catboost_selected_features = borutashap_accepted_features.copy()
if not catboost_selected_features:
    raise RuntimeError("BorutaSHAP selected zero accepted CatBoost features; tentative features are reported but not auto-included.")

selected_features["catboost"] = catboost_selected_features
selected_catboost_categorical_features = [col for col in catboost_categorical_features if col in catboost_selected_features]

borutashap_importance = borutashap_selector.history_x.drop(columns=["Max_Shadow", "Min_Shadow", "Mean_Shadow", "Median_Shadow"], errors="ignore")
borutashap_importance_summary = pd.DataFrame(
    {
        "feature": list(borutashap_importance.columns),
        "mean_importance": borutashap_importance.mean(axis=0).values,
        "median_importance": borutashap_importance.median(axis=0).values,
        "max_importance": borutashap_importance.max(axis=0).values,
    }
)
borutashap_status_map = {
    **{feature: "accepted" for feature in borutashap_accepted_features},
    **{feature: "tentative" for feature in borutashap_tentative_features},
    **{feature: "rejected" for feature in borutashap_rejected_features},
}
borutashap_feature_status = pd.DataFrame(
    [{"feature": feature, "borutashap_status": borutashap_status_map.get(feature, "unassigned")} for feature in X_train_catboost.columns]
).merge(borutashap_importance_summary, on="feature", how="left")
borutashap_feature_status["used_in_sensitivity_catboost"] = borutashap_feature_status["feature"].isin(catboost_selected_features)
borutashap_feature_status = borutashap_feature_status.sort_values(
    ["used_in_sensitivity_catboost", "median_importance"], ascending=[False, False]
).reset_index(drop=True)

catboost_final_selected_features_table = pd.DataFrame({"selected_feature": catboost_selected_features})
borutashap_tentative_features_table = pd.DataFrame({"tentative_feature": borutashap_tentative_features})
borutashap_rejected_features_table = pd.DataFrame({"rejected_feature": borutashap_rejected_features})

feature_selection_details.append(
    {
        "model": "CatBoost",
        "method": FEATURE_SELECTION_METHODS["catboost"],
        "selected_count": len(catboost_selected_features),
        "details": f"Accepted={len(borutashap_accepted_features)}; tentative={len(borutashap_tentative_features)}; rejected={len(borutashap_rejected_features)}; selected categorical={len(selected_catboost_categorical_features)}",
    }
)

print(f"BorutaSHAP accepted CatBoost features: {len(borutashap_accepted_features)}")
print(f"BorutaSHAP tentative CatBoost features: {len(borutashap_tentative_features)}")
print(f"BorutaSHAP rejected CatBoost features: {len(borutashap_rejected_features)}")
display(borutashap_feature_status)
display(catboost_final_selected_features_table)
display(borutashap_tentative_features_table)
display(borutashap_rejected_features_table)

selected_feature_counts = {key: len(value) for key, value in selected_features.items()}
feature_selection_summary = pd.DataFrame(feature_selection_details)
selected_feature_table = pd.DataFrame(
    [{"model": model, "selected_feature": feature} for model, features in selected_features.items() for feature in features]
)

display(feature_selection_summary)
display(selected_feature_table)
display(borutashap_feature_status.head(15))

# 3. Model Benchmarking

This section benchmarks selected-feature models using training-only cross-validation. The held-out test set remains reserved for the final frozen model evaluation.

## 3.1 Evaluation Metrics Definition

F1-score and abnormal EEG recall/sensitivity are primary metrics. Accuracy is reported only as a secondary metric.

In [ ]:
def binary_metrics(y_true, y_score, threshold=0.5):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    y_score = np.asarray(y_score, dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    try:
        roc_auc = roc_auc_score(y_true, y_score)
    except ValueError:
        roc_auc = np.nan
    return {
        "threshold": float(threshold),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def selected_matrix(model_key, split):
    if model_key in ["logistic_regression", "linear_svm"]:
        base = X_train_linear if split == "train" else X_test_linear
    elif model_key == "random_forest":
        base = X_train_tree if split == "train" else X_test_tree
    elif model_key == "catboost":
        base = X_train_catboost if split == "train" else X_test_catboost
    else:
        raise KeyError(model_key)
    return base[selected_features[model_key]].copy()


def make_model(model_key, params=None):
    params = params or {}
    if model_key == "logistic_regression":
        return LogisticRegression(C=params.get("C", 1.0), solver="liblinear", class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
    if model_key == "linear_svm":
        base_svm = LinearSVC(C=params.get("C", 1.0), penalty="l2", class_weight="balanced", max_iter=30000, dual="auto", random_state=RANDOM_STATE)
        return CalibratedClassifierCV(estimator=base_svm, cv=3)
    if model_key == "random_forest":
        return RandomForestClassifier(
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", None),
            min_samples_leaf=params.get("min_samples_leaf", 1),
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    if model_key == "catboost":
        return CatBoostClassifier(
            iterations=params.get("iterations", 200),
            depth=params.get("depth", 4),
            learning_rate=params.get("learning_rate", 0.05),
            l2_leaf_reg=params.get("l2_leaf_reg", 3),
            loss_function="Logloss",
            eval_metric="F1",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )
    raise KeyError(model_key)


def cat_features_for_matrix(model_key, X_matrix):
    if model_key != "catboost":
        return []
    return [col for col in selected_catboost_categorical_features if col in X_matrix.columns]


def fit_model(model, model_key, X_fit, y_fit):
    if model_key == "catboost":
        model.fit(X_fit, y_fit, cat_features=cat_features_for_matrix(model_key, X_fit), verbose=False)
    else:
        model.fit(X_fit, y_fit)
    return model


def predict_scores(model, X_score):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X_score))[:, 1]
    decision = np.asarray(model.decision_function(X_score), dtype=float)
    return 1.0 / (1.0 + np.exp(-decision))


def oof_scores(model_key, X_matrix, y_values, params=None):
    y_values = pd.Series(y_values).astype(int)
    scores = pd.Series(index=y_values.index, dtype=float)
    for train_idx, valid_idx in cv.split(X_matrix, y_values):
        model = make_model(model_key, params=params)
        fit_model(model, model_key, X_matrix.iloc[train_idx].copy(), y_values.iloc[train_idx].copy())
        scores.iloc[valid_idx] = predict_scores(model, X_matrix.iloc[valid_idx].copy())
    return scores.astype(float)


metric_priority = ["f1", "recall_sensitivity", "balanced_accuracy"]
print("Metric helpers and model factories are ready.")

## 3.2 Modelling

The benchmark compares only selected-feature models using out-of-fold training predictions.

In [ ]:
MODEL_DISPLAY_NAMES = {
    "logistic_regression": "Logistic Regression",
    "linear_svm": "Linear SVM",
    "random_forest": "Random Forest",
    "catboost": "CatBoost",
}

benchmark_rows = []
benchmark_oof_scores = {}
for model_key in ["logistic_regression", "linear_svm", "random_forest", "catboost"]:
    X_model_train = selected_matrix(model_key, "train")
    scores = oof_scores(model_key, X_model_train, y_train)
    benchmark_oof_scores[model_key] = scores
    metrics = binary_metrics(y_train, scores, threshold=0.5)
    metrics.update(
        {
            "model_key": model_key,
            "model": MODEL_DISPLAY_NAMES[model_key],
            "feature_selection_method": FEATURE_SELECTION_METHODS[model_key],
            "selected_feature_count": len(selected_features[model_key]),
        }
    )
    benchmark_rows.append(metrics)

benchmark_results = pd.DataFrame(benchmark_rows)[
    [
        "model_key",
        "model",
        "selected_feature_count",
        "f1",
        "recall_sensitivity",
        "precision",
        "specificity",
        "balanced_accuracy",
        "accuracy",
        "roc_auc",
        "tn",
        "fp",
        "fn",
        "tp",
        "feature_selection_method",
    ]
].sort_values(metric_priority, ascending=False)

FINAL_MODEL_KEY = "catboost"
FINAL_MODEL_NAME = MODEL_DISPLAY_NAMES[FINAL_MODEL_KEY]
FINAL_MODEL_SELECTION_REASON = "CatBoost retained as final model because BorutaSHAP all-relevant ML risk-factor discovery is the project focus."

display(benchmark_results)
print(f"{SENSITIVITY_LABEL}")
print(f"Selected final model for train-only tuning: {FINAL_MODEL_NAME}")
print(FINAL_MODEL_SELECTION_REASON)

# 4. Final Model Tuning

This section tunes the selected final model on training data only, tunes the threshold with training out-of-fold predictions only, and then evaluates the frozen model once on the held-out test set.

## 4.1 Final Model Tuning

The final model is tuned after feature selection using stratified cross-validation on the training set.

In [ ]:
TUNING_GRIDS = {
    "logistic_regression": {"C": [0.1, 1.0, 10.0]},
    "linear_svm": {"C": [0.1, 1.0, 10.0]},
    "random_forest": {"n_estimators": [500], "max_depth": [None, 5], "min_samples_leaf": [1, 3]},
    "catboost": {"iterations": [150, 250], "depth": [3, 4], "learning_rate": [0.03], "l2_leaf_reg": [3]},
}

X_train_final = selected_matrix(FINAL_MODEL_KEY, "train")
X_test_final = selected_matrix(FINAL_MODEL_KEY, "test")

tuning_rows = []
tuning_oof_by_params = []
for params in ParameterGrid(TUNING_GRIDS[FINAL_MODEL_KEY]):
    scores = oof_scores(FINAL_MODEL_KEY, X_train_final, y_train, params=params)
    metrics = binary_metrics(y_train, scores, threshold=0.5)
    row = {"model_key": FINAL_MODEL_KEY, "model": FINAL_MODEL_NAME, "params": params}
    row.update(metrics)
    tuning_rows.append(row)
    tuning_oof_by_params.append((params, scores, metrics))

tuning_results = pd.DataFrame(tuning_rows).sort_values(metric_priority, ascending=False).reset_index(drop=True)
best_tuning_params = tuning_results.iloc[0]["params"]
best_tuning_scores = next(scores for params, scores, _ in tuning_oof_by_params if params == best_tuning_params)

best_final_model = make_model(FINAL_MODEL_KEY, params=best_tuning_params)
fit_model(best_final_model, FINAL_MODEL_KEY, X_train_final, y_train)

print(f"Final model selected for tuning: {FINAL_MODEL_NAME}")
print(f"Best train-CV tuning parameters: {best_tuning_params}")
display(tuning_results)

## 4.2 Threshold Tuning

The classification threshold is selected with out-of-fold training predictions only. The held-out test labels are not used for threshold selection.

In [ ]:
def threshold_table(y_true, y_score, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.01), 2)
    return pd.DataFrame([binary_metrics(y_true, y_score, threshold=threshold) for threshold in thresholds])


threshold_results = threshold_table(y_train, best_tuning_scores)
threshold_results = threshold_results.sort_values(metric_priority, ascending=False).reset_index(drop=True)
selected_threshold = float(threshold_results.iloc[0]["threshold"])
threshold_selection_metrics = threshold_results.iloc[0].to_dict()

print(f"Selected threshold from training OOF predictions only: {selected_threshold:.2f}")
display(threshold_results.head(15))

## 4.3 Final Held-Out Test Evaluation

**Sensitivity model, not main model.** The held-out results below describe the length-of-hospitalisation sensitivity CatBoost workflow and must not replace the main deployment-valid model.


In [ ]:
test_scores = predict_scores(best_final_model, X_test_final)
final_test_metrics = binary_metrics(y_test, test_scores, threshold=selected_threshold)
final_test_results = pd.DataFrame([{**{"analysis_label": SENSITIVITY_LABEL, "model": FINAL_MODEL_NAME, "params": best_tuning_params}, **final_test_metrics}])

previous_catboost_reference = {
    "selected_feature_count": 10,
    "threshold": 0.30,
    "f1": 0.835,
    "recall_sensitivity": 0.905,
    "precision": 0.776,
    "specificity": 0.312,
    "balanced_accuracy": 0.609,
    "accuracy": 0.741,
    "roc_auc": 0.766,
    "tn": 5,
    "fp": 11,
    "fn": 4,
    "tp": 38,
}
current_catboost_summary = {
    "selected_feature_count": len(catboost_selected_features),
    **final_test_metrics,
}
catboost_metric_changes_vs_previous = pd.DataFrame(
    [
        {
            "metric": metric,
            "previous_catboost": previous_value,
            "current_catboost": current_catboost_summary.get(metric, selected_threshold if metric == "threshold" else np.nan),
            "change": current_catboost_summary.get(metric, selected_threshold if metric == "threshold" else np.nan) - previous_value,
        }
        for metric, previous_value in previous_catboost_reference.items()
    ]
)
final_confusion_matrix = pd.DataFrame(
    [[final_test_metrics["tn"], final_test_metrics["fp"]], [final_test_metrics["fn"], final_test_metrics["tp"]]],
    index=["Actual normal", "Actual abnormal"],
    columns=["Predicted normal", "Predicted abnormal"],
)

print(f"Final held-out test evaluation complete: {SENSITIVITY_LABEL}. This is the first metric evaluation using held-out test labels for this sensitivity notebook.")
display(final_test_results)
display(final_confusion_matrix)
print("Metric changes versus previous CatBoost reference result:")
display(catboost_metric_changes_vs_previous)

final_test_results.to_csv(OUTPUT_DIR / "final_held_out_test_metrics.csv", index=False)
final_test_results.to_csv(OUTPUT_DIR / "final_catboost_metrics.csv", index=False)
final_confusion_matrix.to_csv(OUTPUT_DIR / "final_catboost_confusion_matrix.csv")
benchmark_results.to_csv(OUTPUT_DIR / "benchmark_results_train_cv.csv", index=False)
feature_selection_summary.to_csv(OUTPUT_DIR / "feature_selection_summary.csv", index=False)
selected_feature_table.to_csv(OUTPUT_DIR / "selected_features.csv", index=False)
borutashap_feature_status.to_csv(OUTPUT_DIR / "catboost_borutashap_feature_status.csv", index=False)
catboost_final_selected_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_accepted_features.csv", index=False)
borutashap_tentative_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_tentative_features.csv", index=False)
borutashap_rejected_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_rejected_features.csv", index=False)
catboost_metric_changes_vs_previous.to_csv(OUTPUT_DIR / "catboost_metric_changes_vs_previous.csv", index=False)

# Compare the sensitivity CatBoost result against the current main CatBoost outputs saved by the main notebook.
comparison_metrics = ["f1", "recall_sensitivity", "precision", "specificity", "balanced_accuracy", "accuracy", "roc_auc"]
main_metrics_path = MAIN_OUTPUT_DIR / "final_held_out_test_metrics.csv"
main_selected_features_path = MAIN_OUTPUT_DIR / "selected_features.csv"
if main_metrics_path.exists():
    main_final_metrics = pd.read_csv(main_metrics_path).iloc[0].to_dict()
else:
    main_final_metrics = {}
    print(f"Main CatBoost metrics file not found for comparison: {main_metrics_path}")

main_vs_sensitivity_metrics = pd.DataFrame(
    [
        {
            "metric": metric,
            "main_catboost": main_final_metrics.get(metric, np.nan),
            "sensitivity_catboost": final_test_metrics.get(metric, np.nan),
            "sensitivity_minus_main": final_test_metrics.get(metric, np.nan) - main_final_metrics.get(metric, np.nan)
            if metric in main_final_metrics
            else np.nan,
        }
        for metric in comparison_metrics
    ]
)

confusion_metric_names = ["tn", "fp", "fn", "tp"]
main_vs_sensitivity_confusion = pd.DataFrame(
    [
        {
            "model": "Main CatBoost",
            **{metric: main_final_metrics.get(metric, np.nan) for metric in confusion_metric_names},
        },
        {
            "model": "Sensitivity CatBoost: LENGTH OF HOSPITALISATION candidate",
            **{metric: final_test_metrics.get(metric, np.nan) for metric in confusion_metric_names},
        },
    ]
)

if main_selected_features_path.exists():
    main_selected_feature_table = pd.read_csv(main_selected_features_path)
    main_catboost_features = main_selected_feature_table.loc[
        main_selected_feature_table["model"].eq("catboost"), "selected_feature"
    ].dropna().astype(str).tolist()
else:
    main_catboost_features = []
    print(f"Main selected-features file not found for comparison: {main_selected_features_path}")

sensitivity_catboost_features = list(selected_features["catboost"])
selected_feature_comparison = pd.DataFrame(
    [
        {
            "feature": feature,
            "in_main_catboost": feature in main_catboost_features,
            "in_sensitivity_catboost": feature in sensitivity_catboost_features,
        }
        for feature in sorted(set(main_catboost_features + sensitivity_catboost_features + [SENSITIVITY_FEATURE]))
    ]
)

print("Sensitivity CatBoost versus current main CatBoost metrics")
display(main_vs_sensitivity_metrics)
print("Sensitivity CatBoost versus current main CatBoost confusion matrix counts")
display(main_vs_sensitivity_confusion)
print("Selected feature comparison")
display(selected_feature_comparison)

main_vs_sensitivity_metrics.to_csv(OUTPUT_DIR / "catboost_metrics_vs_main.csv", index=False)
main_vs_sensitivity_confusion.to_csv(OUTPUT_DIR / "catboost_confusion_matrix_vs_main.csv", index=False)
selected_feature_comparison.to_csv(OUTPUT_DIR / "catboost_selected_features_vs_main.csv", index=False)


# 5. SHAP Analysis

This section explains the frozen final sensitivity model. **Sensitivity model, not main model.** SHAP explains model behavior, not causality.


## 5.1 Global SHAP Explanation

Global SHAP bar and beeswarm plots are generated for the frozen sensitivity CatBoost model using the BorutaSHAP accepted feature set. BorutaSHAP was used for all-relevant ML risk-factor discovery; SHAP explains how the final CatBoost model used those selected features.

In [ ]:
def numeric_display_frame(X_frame):
    display_frame = X_frame.copy()
    category_maps = {}
    for col in display_frame.columns:
        if not pd.api.types.is_numeric_dtype(display_frame[col]):
            cat = pd.Categorical(display_frame[col].astype(str))
            category_maps[col] = dict(enumerate(cat.categories))
            display_frame[col] = cat.codes.astype(float)
    return display_frame.astype(float), category_maps


def build_shap_explanation(model_key, model, X_background, X_explain):
    feature_names = list(X_explain.columns)
    X_display, category_maps = numeric_display_frame(X_explain)
    if model_key == "catboost":
        pool = Pool(X_explain, cat_features=cat_features_for_matrix(model_key, X_explain))
        raw_shap = model.get_feature_importance(pool, type="ShapValues")
        explanation = shap.Explanation(
            values=raw_shap[:, :-1],
            base_values=raw_shap[:, -1],
            data=X_display.to_numpy(),
            feature_names=feature_names,
        )
        return explanation, category_maps

    if model_key == "random_forest":
        explainer = shap.TreeExplainer(model)
        raw = explainer.shap_values(X_explain)
        if isinstance(raw, list):
            values = raw[1]
            base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
        elif np.asarray(raw).ndim == 3:
            values = np.asarray(raw)[:, :, 1]
            base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
        else:
            values = np.asarray(raw)
            base = explainer.expected_value
        explanation = shap.Explanation(
            values=values,
            base_values=np.repeat(base, len(X_explain)),
            data=X_display.to_numpy(),
            feature_names=feature_names,
        )
        return explanation, category_maps

    background = X_background.sample(min(100, len(X_background)), random_state=RANDOM_STATE)

    def predict_positive(values):
        return predict_scores(model, pd.DataFrame(values, columns=feature_names))

    explainer = shap.Explainer(
        predict_positive,
        shap.maskers.Independent(background),
        algorithm="permutation",
        feature_names=feature_names,
    )
    explanation = explainer(X_explain, max_evals=max(2 * len(feature_names) + 1, 50))
    values = explanation.values
    base_values = explanation.base_values
    if np.asarray(values).ndim == 3:
        values = np.asarray(values)[:, :, 1]
        base_values = np.asarray(base_values)[:, 1]
    return shap.Explanation(values=values, base_values=base_values, data=X_display.to_numpy(), feature_names=feature_names), category_maps


X_shap_background = X_train_final.copy()
X_shap_explain = X_test_final.copy()
shap_explanation, shap_category_maps = build_shap_explanation(FINAL_MODEL_KEY, best_final_model, X_shap_background, X_shap_explain)

shap_importance = pd.DataFrame(
    {"feature": list(X_shap_explain.columns), "mean_abs_shap": np.abs(shap_explanation.values).mean(axis=0)}
).sort_values("mean_abs_shap", ascending=False)
top_shap_features = list(shap_importance.head(5)["feature"])

plt.figure(figsize=(8, 6))
shap.plots.bar(shap_explanation, max_display=min(15, len(X_shap_explain.columns)), show=False)
plt.tight_layout()
shap_bar_path = SHAP_OUTPUT_DIR / "global_shap_bar.png"
plt.savefig(shap_bar_path, dpi=200, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 6))
shap.plots.beeswarm(shap_explanation, max_display=min(15, len(X_shap_explain.columns)), show=False)
plt.tight_layout()
shap_beeswarm_path = SHAP_OUTPUT_DIR / "global_shap_beeswarm.png"
plt.savefig(shap_beeswarm_path, dpi=200, bbox_inches="tight")
plt.close()

shap_importance.to_csv(SHAP_OUTPUT_DIR / "shap_feature_importance.csv", index=False)
print("Global SHAP outputs produced:")
print(shap_bar_path)
print(shap_beeswarm_path)
display(shap_importance.head(15))

## 5.2 SHAP Dependence Plots

Dependence plots are generated for the top SHAP-ranked BorutaSHAP-selected features where feasible. These are model-associated predictors/risk factors, not causal claims.

In [ ]:
dependence_plot_paths = []
shap_display_df = pd.DataFrame(shap_explanation.data, columns=list(X_shap_explain.columns), index=X_shap_explain.index)


def save_dependence_plot(feature, explanation, display_df, output_dir, label_suffix=""):
    plt.figure(figsize=(7, 5))
    shap.dependence_plot(
        feature,
        explanation.values,
        display_df,
        feature_names=list(display_df.columns),
        show=False,
        interaction_index=None,
    )
    plt.title(f"{feature} SHAP dependence{label_suffix}\nSensitivity model, not main model")
    plt.tight_layout()
    safe_feature = "".join(ch if ch.isalnum() else "_" for ch in feature).strip("_")[:80]
    path = output_dir / f"dependence_{safe_feature}.png"
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    return path


for feature in top_shap_features[:5]:
    try:
        path = save_dependence_plot(feature, shap_explanation, shap_display_df, SHAP_OUTPUT_DIR)
        dependence_plot_paths.append(path)
    except Exception as exc:
        plt.close()
        print(f"Skipped dependence plot for {feature}: {exc}")

hospitalisation_dependence_plot_path = None
hospitalisation_dependence_plot_source = None
hospitalisation_shap_values = np.array([], dtype=float)
hospitalisation_display_values = np.array([], dtype=float)

if SENSITIVITY_FEATURE in X_shap_explain.columns:
    hospitalisation_explanation = shap_explanation
    hospitalisation_display_df = shap_display_df
    hospitalisation_dependence_plot_source = "Final sensitivity CatBoost model"
elif SENSITIVITY_FEATURE in X_test_catboost.columns:
    inspection_features = list(dict.fromkeys(list(X_shap_explain.columns) + [SENSITIVITY_FEATURE]))
    X_train_hospitalisation_inspection = X_train_catboost[inspection_features].copy()
    X_test_hospitalisation_inspection = X_test_catboost[inspection_features].copy()
    hospitalisation_inspection_model = make_model("catboost", params=best_tuning_params)
    fit_model(hospitalisation_inspection_model, "catboost", X_train_hospitalisation_inspection, y_train)
    hospitalisation_explanation, hospitalisation_category_maps = build_shap_explanation(
        "catboost",
        hospitalisation_inspection_model,
        X_train_hospitalisation_inspection,
        X_test_hospitalisation_inspection,
    )
    hospitalisation_display_df = pd.DataFrame(
        hospitalisation_explanation.data,
        columns=list(X_test_hospitalisation_inspection.columns),
        index=X_test_hospitalisation_inspection.index,
    )
    hospitalisation_dependence_plot_source = "Inspection CatBoost model using final selected features plus LENGTH OF HOSPITALISATION"
else:
    hospitalisation_explanation = None
    hospitalisation_display_df = None
    print(f"{SENSITIVITY_FEATURE} is unavailable after preprocessing; dedicated dependence plot cannot be generated.")

if hospitalisation_explanation is not None and SENSITIVITY_FEATURE in hospitalisation_display_df.columns:
    try:
        hospitalisation_dependence_plot_path = save_dependence_plot(
            SENSITIVITY_FEATURE,
            hospitalisation_explanation,
            hospitalisation_display_df,
            SHAP_OUTPUT_DIR,
            label_suffix=" (hospitalisation sensitivity)",
        )
        if hospitalisation_dependence_plot_path not in dependence_plot_paths:
            dependence_plot_paths.append(hospitalisation_dependence_plot_path)
        feature_index = list(hospitalisation_display_df.columns).index(SENSITIVITY_FEATURE)
        hospitalisation_shap_values = np.asarray(hospitalisation_explanation.values)[:, feature_index]
        hospitalisation_display_values = hospitalisation_display_df[SENSITIVITY_FEATURE].astype(float).to_numpy()
    except Exception as exc:
        plt.close()
        print(f"Skipped dedicated dependence plot for {SENSITIVITY_FEATURE}: {exc}")


def estimate_positive_shap_threshold(feature_values, shap_values):
    signal = pd.DataFrame({"feature_value": feature_values, "shap_value": shap_values}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(signal) < 3:
        return np.nan
    signal = signal.sort_values("feature_value").reset_index(drop=True)
    window = max(3, min(9, int(np.ceil(len(signal) * 0.15))))
    signal["rolling_median_shap"] = signal["shap_value"].rolling(window=window, min_periods=2, center=True).median()
    positive_signal = signal.loc[signal["rolling_median_shap"] > 0, "feature_value"]
    if positive_signal.empty:
        return np.nan
    return float(positive_signal.iloc[0])

hospitalisation_borutashap_status = "not_in_borutashap_table"
if "borutashap_feature_status" in globals() and SENSITIVITY_FEATURE in set(borutashap_feature_status["feature"]):
    hospitalisation_borutashap_status = str(
        borutashap_feature_status.loc[
            borutashap_feature_status["feature"].eq(SENSITIVITY_FEATURE), "borutashap_status"
        ].iloc[0]
    )

hospitalisation_positive_shap_threshold = estimate_positive_shap_threshold(
    hospitalisation_display_values,
    hospitalisation_shap_values,
)

hospitalisation_shap_signal_summary = pd.DataFrame(
    [
        {
            "feature": SENSITIVITY_FEATURE,
            "borutashap_status": hospitalisation_borutashap_status,
            "in_final_sensitivity_catboost": SENSITIVITY_FEATURE in list(X_shap_explain.columns),
            "dependence_plot_source": hospitalisation_dependence_plot_source,
            "approx_positive_shap_threshold_days": hospitalisation_positive_shap_threshold,
            "mean_shap": float(np.nanmean(hospitalisation_shap_values)) if len(hospitalisation_shap_values) else np.nan,
            "mean_abs_shap": float(np.nanmean(np.abs(hospitalisation_shap_values))) if len(hospitalisation_shap_values) else np.nan,
            "plot_path": str(hospitalisation_dependence_plot_path) if hospitalisation_dependence_plot_path else None,
            "interpretation": "Hospital-course / illness-severity proxy only; main model should still exclude it.",
        }
    ]
)
hospitalisation_shap_signal_summary.to_csv(OUTPUT_DIR / "length_of_hospitalisation_shap_signal_summary.csv", index=False)

print("Dependence plot outputs:")
for path in dependence_plot_paths:
    print(path)
print("Length of hospitalisation SHAP signal summary")
display(hospitalisation_shap_signal_summary)


## 5.3 Local SHAP Explanation

A de-identified local waterfall plot is generated for one held-out example where feasible. No direct identifiers are included. The local explanation describes model behavior for the selected features, not clinical causality.

In [ ]:
waterfall_path = None
try:
    plt.figure(figsize=(8, 6))
    shap.plots.waterfall(shap_explanation[0], max_display=min(15, len(X_shap_explain.columns)), show=False)
    plt.tight_layout()
    waterfall_path = SHAP_OUTPUT_DIR / "local_waterfall_example_held_out_case.png"
    plt.savefig(waterfall_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Local waterfall output: {waterfall_path}")
except Exception as exc:
    plt.close()
    print(f"Waterfall plot was not feasible: {exc}")

## 5.4 Clinical Interpretation, Limitations, and Final Summary

This final summary is for the length-of-hospitalisation sensitivity analysis only. **Sensitivity model, not main model.** `LENGTH OF HOSPITALISATION` is interpreted only as a possible illness-severity or hospital-course proxy, not as a clean pre-EEG predictor.


In [ ]:
shap_outputs = {
    "global_bar": str(shap_bar_path),
    "global_beeswarm": str(shap_beeswarm_path),
    "dependence_plots": [str(path) for path in dependence_plot_paths],
    "waterfall": str(waterfall_path) if waterfall_path else None,
    "importance_csv": str(SHAP_OUTPUT_DIR / "shap_feature_importance.csv"),
}

final_summary = {
    "project": PROJECT_NAME,
    "analysis_label": SENSITIVITY_LABEL,
    "final_model": FINAL_MODEL_NAME,
    "selected_threshold": selected_threshold,
    "selected_feature_count": len(selected_features[FINAL_MODEL_KEY]),
    "borutashap_accepted_features": borutashap_accepted_features,
    "borutashap_tentative_features": borutashap_tentative_features,
    "final_test_f1": final_test_metrics["f1"],
    "final_test_recall_sensitivity": final_test_metrics["recall_sensitivity"],
    "final_test_precision": final_test_metrics["precision"],
    "final_test_specificity": final_test_metrics["specificity"],
    "final_test_balanced_accuracy": final_test_metrics["balanced_accuracy"],
    "final_test_roc_auc": final_test_metrics["roc_auc"],
    "top_shap_features": top_shap_features,
    "length_of_hospitalisation_borutashap_status": hospitalisation_borutashap_status,
    "length_of_hospitalisation_in_final_sensitivity_catboost": SENSITIVITY_FEATURE in list(X_shap_explain.columns),
    "length_of_hospitalisation_positive_shap_threshold_days": hospitalisation_positive_shap_threshold,
    "length_of_hospitalisation_dependence_plot_source": hospitalisation_dependence_plot_source,
    "research_only_note": RESEARCH_ONLY_NOTE,
}

manuscript_risk_factor_mapping = {
    "seizures": ["SEIZURES", "FOCAL SEIZURE", "GENERALISED SEIZURE"],
    "hyperlipidaemia": ["HYPERLIPIDEMIA"],
    "infection": ["WHITE CELL COUNT"],
    "structural causes": ["RADIOLOGICAL IMAGING", "BRAIN TUMOUR", "STROKE"],
    "stroke": ["STROKE"],
    "radiological infarct": ["RADIOLOGICAL IMAGING"],
    "creatinine": ["CREATININE"],
    "albumin": ["ALBUMIN"],
    "length of hospitalisation": ["LENGTH OF HOSPITALISATION"],
}

manuscript_risk_factor_comparison = pd.DataFrame(
    [
        {
            "manuscript_risk_factor": risk_factor,
            "candidate_or_related_features": features,
            "borutashap_accepted_overlap": [feature for feature in features if feature in borutashap_accepted_features],
            "borutashap_tentative_overlap": [feature for feature in features if feature in borutashap_tentative_features],
            "borutashap_rejected_overlap": [feature for feature in features if feature in borutashap_rejected_features],
            "notes": "Intentionally included only in sensitivity analysis; main model should still exclude it" if "LENGTH OF HOSPITALISATION" in features else "Compared against retained candidate feature pool",
        }
        for risk_factor, features in manuscript_risk_factor_mapping.items()
    ]
)

clinical_caveats = [
    "BorutaSHAP was used for all-relevant ML risk-factor discovery with CatBoost on training data only.",
    "SHAP explains how the final CatBoost model used the BorutaSHAP-selected features; it does not establish causality.",
    "Selected features are model-associated predictors/risk factors, not causal claims or diagnostic evidence by themselves.",
    "Seizure-related predictors require timing confirmation relative to EEG ordering.",
    "Blood Sugar Control availability and clinical meaning require confirmation.",
    "Missingness flags may reflect testing or documentation patterns rather than biology.",
    "The dataset is small, single-centre, retrospective, and elderly; external validation is required before any clinical use.",
    "LENGTH OF HOSPITALISATION is a possible illness-severity or hospital-course proxy, not a clean pre-EEG predictor.",
    "The main model should still exclude LENGTH OF HOSPITALISATION because it may represent post-admission/post-outcome leakage.",
    "No clinical deployment, diagnostic decision support, or treatment recommendation claims are made.",
]

pd.Series(final_summary).to_frame("value").to_csv(OUTPUT_DIR / "final_summary.csv")
manuscript_risk_factor_comparison.to_csv(OUTPUT_DIR / "manuscript_vs_borutashap_risk_factor_comparison.csv", index=False)
pd.Series(shap_outputs).to_frame("path").to_csv(SHAP_OUTPUT_DIR / "shap_outputs.csv")

threshold_text = (
    f"approximately {hospitalisation_positive_shap_threshold:.2f} days"
    if pd.notna(hospitalisation_positive_shap_threshold)
    else "not identified from the available SHAP dependence signal"
)
main_feature_text = ", ".join(main_catboost_features) if main_catboost_features else "Unavailable from current main outputs"
sensitivity_feature_text = ", ".join(sensitivity_catboost_features) if sensitivity_catboost_features else "None"


def dataframe_to_markdown(df):
    columns = list(df.columns)
    def fmt(value):
        if pd.isna(value):
            return ""
        if isinstance(value, float):
            return f"{value:.6g}"
        return str(value).replace("|", "\|")
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = ["| " + " | ".join(fmt(row[col]) for col in columns) + " |" for _, row in df.iterrows()]
    return "\n".join([header, separator] + rows)

metrics_markdown = dataframe_to_markdown(main_vs_sensitivity_metrics)
confusion_markdown = dataframe_to_markdown(main_vs_sensitivity_confusion)
conclusion_markdown = f"""# Length of Hospitalisation Sensitivity Analysis Conclusion

**Sensitivity model, not main model.**

`LENGTH OF HOSPITALISATION` was intentionally retained as a candidate predictor in this sensitivity notebook only. It must be interpreted as a possible illness-severity or hospital-course proxy, not as a clean pre-EEG predictor. The main EEG abnormality model should still exclude it because it may represent post-admission/post-outcome leakage.

## BorutaSHAP Status

- `LENGTH OF HOSPITALISATION`: **{hospitalisation_borutashap_status}**.
- Included in final sensitivity CatBoost selected feature set: **{SENSITIVITY_FEATURE in list(X_shap_explain.columns)}**.
- Dedicated SHAP dependence plot source: {hospitalisation_dependence_plot_source}.
- Approximate threshold where its SHAP contribution begins pushing prediction toward abnormal EEG: **{threshold_text}**.

## CatBoost Held-Out Test Comparison

{metrics_markdown}

## Confusion Matrix Comparison

{confusion_markdown}

## Selected Features

- Main CatBoost selected features: {main_feature_text}
- Sensitivity CatBoost selected features: {sensitivity_feature_text}

## Interpretation

This sensitivity analysis explores whether hospitalisation length behaves like an illness-course signal in model explanations. Even if selected or visually informative in SHAP, it should not be promoted into the main deployment-valid prediction model without a revised clinical question and timing justification.

Research-only limitation: this is a small, single-centre, retrospective dataset and the outputs are not medical advice, diagnostic decision support, or deployment-ready evidence.
"""
(OUTPUT_DIR / "length_of_hospitalisation_sensitivity_conclusion.md").write_text(conclusion_markdown, encoding="utf-8")

print(f"Final model summary: {SENSITIVITY_LABEL}")
display(pd.Series(final_summary).to_frame("value"))
print("SHAP outputs")
display(pd.Series(shap_outputs).to_frame("path"))
print("Manuscript risk-factor comparison")
display(manuscript_risk_factor_comparison)
print("Clinical caveats and limitations")
for caveat in clinical_caveats:
    print(f"- {caveat}")